In [2]:
import glob
import json
import os
from pathlib import Path

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from shapely.geometry import Point
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

# =====================================================================
# 1. RAW INGESTION, DOMAIN FILTERING & LEAKAGE ELIMINATION
# =====================================================================

# NEW: Explicit data loading, SFR-only filtering, and leakage scrubbing before modeling.
def load_and_clean_crmls_data(data_folder: str = "./data") -> pd.DataFrame:
    """Loads raw CRMLS monthly exports, filters SFR subset, and eliminates target leakage."""
    file_list = sorted(glob.glob(os.path.join(data_folder, "CRMLSSold*.csv")))
    if len(file_list) < 6:
        print(f"[Warning] Found {len(file_list)} files. Ensure >=6 months for backtesting.")

    df_raw = pd.concat([pd.read_csv(f, low_memory=False) for f in file_list], ignore_index=True)

    # NEW: Restrict the domain to residential single-family homes to match the modeling task.
    df_sfr = df_raw[
        (df_raw["PropertyType"] == "Residential") &
        (df_raw["PropertySubType"] == "SingleFamilyResidence")
    ].copy()

    df_sfr["CloseDate"] = pd.to_datetime(df_sfr["CloseDate"])

    # NEW: Remove implausible property records that would poison the target distribution.
    df_clean = df_sfr[
        (df_sfr["ClosePrice"] > 0) &
        (df_sfr["LivingArea"] > 0) &
        (df_sfr["LotSizeSquareFeet"] > 0) &
        (df_sfr["LotSizeSquareFeet"] >= df_sfr["LivingArea"])
    ].copy()

    # NEW: Drop known target leakage and price-velocity fields that are not causally valid for prediction.
    leakage_cols = [
        "ListPrice", "OriginalListPrice", "PreviousListPrice", "CurrentPrice",
        "DaysOnMarket", "CumulativeDaysOnMarket", "PriceChangeTimestamp",
        "MajorChangeType", "ConcessionsAmount", "ConcessionsComments",
        "BuyerBrokerageCompensation", "BuyerAgencyCompensation",
        "BuyerAgencyCompensationType", "CoBuyerAgentFirstName",
        "CoListAgentFirstName", "CoListAgentLastName", "CoListOfficeName",
        "latfilled", "lonfilled"
    ]
    df_clean.drop(columns=[c for c in leakage_cols if c in df_clean.columns], inplace=True)

    # NEW: Drop columns with >95% missing values that add noise without predictive value.
    cols_high_missing = [
        "AboveGradeFinishedArea", "TaxAnnualAmount", "FireplacesTotal",
        "MiddleOrJuniorSchoolDistrict", "BusinessType", "CoveredSpaces",
        "TaxYear", "ElementarySchoolDistrict", "BuilderName",
        "LotSizeDimensions", "BuildingAreaTotal", "OriginatingSystemName",
        "OriginatingSystemSubName"
    ]
    df_clean.drop(columns=[c for c in cols_high_missing if c in df_clean.columns], inplace=True)

    # NEW: Encode implicit binary/flag fields and create a missingness flag for HOA fees.
    if "WaterfrontYN" in df_clean.columns:
        df_clean["WaterfrontYN"] = df_clean["WaterfrontYN"].fillna(0).astype(int)
    if "BasementYN" in df_clean.columns:
        df_clean["BasementYN"] = df_clean["BasementYN"].fillna(0).astype(int)
    if "BelowGradeFinishedArea" in df_clean.columns:
        df_clean["BelowGradeFinishedArea"] = df_clean["BelowGradeFinishedArea"].fillna(0.0)
    if "AssociationFee" in df_clean.columns:
        df_clean["AssociationFee_isna"] = df_clean["AssociationFee"].isna().astype(int)
        df_clean["AssociationFee"] = df_clean["AssociationFee"].fillna(0.0)
    if "AssociationFeeFrequency" in df_clean.columns:
        df_clean["AssociationFeeFrequency"] = df_clean["AssociationFeeFrequency"].fillna("None")

    return df_clean


In [9]:
# =====================================================================
# 2. SPATIAL ENRICHMENT & FEATURE ENGINEERING
# =====================================================================

# NEW: Spatially enrich homes with school district labels using geographic joins.
def augment_school_districts(df: pd.DataFrame, school_geojson_path: str) -> pd.DataFrame:
    """Assigns school district labels to properties with valid coordinates."""
    df_spatial = df.copy()
    district_labels = np.full(len(df_spatial), "Unknown", dtype=object)

    valid_positions = np.flatnonzero(
        df_spatial["Latitude"].notna().to_numpy()
        & df_spatial["Longitude"].notna().to_numpy()
    )

    if len(valid_positions) == 0:
        df_spatial["Spatial_SchoolDistrict"] = district_labels
        return df_spatial

    df_valid = df_spatial.iloc[valid_positions][["Latitude", "Longitude"]].reset_index(drop=True)
    geometry = [
        Point(lon, lat)
        for lon, lat in zip(df_valid["Longitude"], df_valid["Latitude"])
    ]
    gdf_properties = gpd.GeoDataFrame(df_valid, geometry=geometry, crs="EPSG:4326")
    gdf_properties["__spatial_row_id"] = np.arange(len(gdf_properties))

    gdf_schools = gpd.read_file(school_geojson_path)
    if gdf_schools.crs is None:
        raise ValueError("The school district file has no CRS defined.")
    if gdf_schools.crs.to_string() != "EPSG:4326":
        gdf_schools = gdf_schools.to_crs("EPSG:4326")

    joined = gpd.sjoin(gdf_properties, gdf_schools, how="left", predicate="within")
    district_col = next(
        (
            col for col in ["DistrictName", "District", "CDCode", "NAME", "SchoolDistrict"]
            if col in joined.columns
        ),
        None,
    )

    if district_col is not None:
        matches = joined.drop_duplicates("__spatial_row_id", keep="first")
        labels = (
            matches.set_index("__spatial_row_id")[district_col]
            .reindex(np.arange(len(valid_positions)))
            .fillna("Unknown")
            .astype(str)
            .to_numpy()
        )
        district_labels[valid_positions] = labels

    df_spatial["Spatial_SchoolDistrict"] = district_labels
    return df_spatial

# NEW: Engineer intrinsic and temporal features while avoiding leakage from the future.
def engineer_avm_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineers intrinsic and temporal features without leakage."""
    df_out = df.copy()

    # NEW: Normalize boolean columns to string for imputer compatibility and stability.
    bool_cols = df_out.select_dtypes(include=["bool"]).columns
    df_out[bool_cols] = df_out[bool_cols].astype(str)

    # NEW: Create time-aware pricing features tied to the transaction date.
    df_out["CloseYear"] = df_out["CloseDate"].dt.year
    df_out["PropertyAge"] = df_out["CloseYear"] - df_out["YearBuilt"]

    # NEW: Price per square foot is kept only for outlier calibration, not as a final model feature.
    df_out["PricePerSqFt"] = df_out["ClosePrice"] / df_out["LivingArea"]

    # NEW: Capture bedroom-to-bathroom intensity while handling zero-bath edge cases.
    bath_safe = df_out["BathroomsTotalInteger"].replace(0, np.nan)
    df_out["BedBathRatio"] = df_out["BedroomsTotal"] / bath_safe

    return df_out


In [4]:
# =====================================================================
# 3. CHRONOLOGICAL SPLITTING & FROZEN TRAIN OUTLIER BOUNDS
# =====================================================================

# NEW: Keep temporal ordering strict to prevent validation/test leakage from future periods.
def get_chronological_splits(data: pd.DataFrame, train_months: int = 6):
    """Splits dataset chronologically based on CloseDate without random shuffling."""
    latest_date = data["CloseDate"].max()
    test_start = latest_date.to_period("M").to_timestamp()
    val_start = (test_start - pd.DateOffset(months=1)).to_period("M").to_timestamp()
    train_start = val_start - pd.DateOffset(months=train_months)

    train_data = data[(data["CloseDate"] >= train_start) & (data["CloseDate"] < val_start)].copy()
    val_data = data[(data["CloseDate"] >= val_start) & (data["CloseDate"] < test_start)].copy()
    test_data = data[data["CloseDate"] >= test_start].copy()

    return train_data, val_data, test_data

# NEW: Freeze outlier thresholds learned only on the training split and apply them consistently.
def apply_frozen_outlier_bounds(
    df: pd.DataFrame, p_low: float, p_high: float, psq_low: float, psq_high: float
) -> pd.DataFrame:
    """Applies frozen train-learned outlier boundaries across all data splits."""
    mask = (
        (df["ClosePrice"] >= p_low) &
        (df["ClosePrice"] <= p_high) &
        (df["PricePerSqFt"] >= psq_low) &
        (df["PricePerSqFt"] <= psq_high)
    )
    return df[mask].copy()


In [5]:
# =====================================================================
# 4. ARTIFACT SERIALIZATION PIPELINE
# =====================================================================

# NEW: Save the fitted preprocessing stage, transformed features, and targets in one export routine.
def export_model_artifacts(
    preprocessor: ColumnTransformer,
    X_train_proc, X_val_proc, X_test_proc,
    y_train: pd.Series, y_val: pd.Series, y_test: pd.Series,
    models: dict = None,
    export_dir: Path = Path("./model_artifacts")
):
    """Serializes fitted preprocessor pipeline, feature matrices, targets, and optional models."""
    export_dir.mkdir(parents=True, exist_ok=True)

    # NEW: Save the fitted scikit-learn preprocessor and output schema.
    joblib.dump(preprocessor, export_dir / "preprocessor.joblib")

    if hasattr(preprocessor, "get_feature_names_out"):
        feature_names = list(preprocessor.get_feature_names_out())
        with open(export_dir / "feature_names.json", "w") as f:
            json.dump(feature_names, f, indent=4)
    else:
        feature_names = None

    # NEW: Export dense/sparse transformed data while preserving feature names where possible.
    def _export_matrix(data, prefix: str):
        if isinstance(data, pd.DataFrame):
            data.to_parquet(export_dir / f"{prefix}.parquet", index=False)
        elif isinstance(data, np.ndarray):
            if feature_names and len(feature_names) == data.shape[1]:
                pd.DataFrame(data, columns=feature_names).to_parquet(export_dir / f"{prefix}.parquet", index=False)
            else:
                np.save(export_dir / f"{prefix}.npy", data)
        elif issparse(data):
            if feature_names and len(feature_names) == data.shape[1]:
                pd.DataFrame(data.toarray(), columns=feature_names).to_parquet(export_dir / f"{prefix}.parquet", index=False)
            else:
                from scipy.sparse import save_npz
                save_npz(export_dir / f"{prefix}.npz", data)

    _export_matrix(X_train_proc, "X_train_processed")
    _export_matrix(X_val_proc, "X_val_processed")
    _export_matrix(X_test_proc, "X_test_processed")

    # NEW: Save the target vectors in parquet format so downstream evaluation can load them cleanly.
    y_train.to_frame("ClosePrice").to_parquet(export_dir / "y_train.parquet")
    y_val.to_frame("ClosePrice").to_parquet(export_dir / "y_val.parquet")
    y_test.to_frame("ClosePrice").to_parquet(export_dir / "y_test.parquet")

    # NEW: Save fitted models when supplied for later comparison or benchmarking.
    if models:
        for name, model in models.items():
            clean_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
            joblib.dump(model, export_dir / f"{clean_name}.joblib")

    print(f"[Success] Data preprocessing artifacts successfully exported to {export_dir.resolve()}")


In [10]:
# =====================================================================
# 5. MAIN PREPROCESSING & ARTIFACT EXPORT PIPELINE
# =====================================================================

# NEW: Build the full preprocessing pipeline from raw monthly CRMLS exports through target-safe artifact export.
if __name__ == "__main__":
    # Step A: Load raw CRMLS data and apply domain + leakage filtering.

    df_clean = load_and_clean_crmls_data(data_folder="./data")
    # Read the already-downloaded school district boundary data.
    district_path = Path("./data/DistrictAreas2425")

    if district_path.is_dir():
        vector_files = [
            p for p in district_path.rglob("*")
            if p.suffix.lower() in {".geojson", ".json", ".shp", ".gpkg"}
        ]
        if len(vector_files) != 1:
            raise FileNotFoundError(
                f"Expected one district boundary file in {district_path}, "
                f"found {len(vector_files)}."
            )
        district_path = vector_files[0]

    if not district_path.exists():
        raise FileNotFoundError(f"District data not found: {district_path}")

    df_augmented = augment_school_districts(df_clean, str(district_path))
    df_augmented = engineer_avm_features(df_augmented)

    # Step B: Chronological split for time-safe validation and test evaluation.
    train_df, val_df, test_df = get_chronological_splits(df_augmented, train_months=6)

    # Step C: Compute frozen outlier thresholds strictly on TRAIN.
    price_p05, price_p995 = train_df["ClosePrice"].quantile([0.005, 0.995])
    ppsqft_p05, ppsqft_p995 = train_df["PricePerSqFt"].quantile([0.005, 0.995])

    train_clean = apply_frozen_outlier_bounds(train_df, price_p05, price_p995, ppsqft_p05, ppsqft_p995)
    val_clean = apply_frozen_outlier_bounds(val_df, price_p05, price_p995, ppsqft_p05, ppsqft_p995)
    test_clean = apply_frozen_outlier_bounds(test_df, price_p05, price_p995, ppsqft_p05, ppsqft_p995)

    # Step D: Define feature taxonomy while excluding leakage and derived price-per-sqft from model inputs.
    num_features = [
        "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
        "LotSizeSquareFeet", "PropertyAge", "AssociationFee", "BedBathRatio"
    ]
    low_card_cat_features = [
        c for c in ["WaterfrontYN", "BasementYN", "AssociationFeeFrequency", "PropertyCondition"]
        if c in train_clean.columns
    ]
    high_card_cat_features = [
        c for c in ["SubdivisionName", "City", "PostalCode", "Spatial_SchoolDistrict"]
        if c in train_clean.columns
    ]

    # Step E: Build the scikit-learn preprocessing pipeline with median imputation + scaling for numeric columns.
    num_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])

    low_card_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    high_card_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target_enc", TargetEncoder(smooth="auto", cv=5, target_type="continuous")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_transformer, num_features),
            ("low_card", low_card_transformer, low_card_cat_features),
            ("high_card", high_card_transformer, high_card_cat_features),
        ],
        remainder="drop",
    )

    # Step F: Fit the preprocessor on the log-transformed target to stabilize price variance.
    y_train_log = np.log1p(train_clean["ClosePrice"])

    X_train_proc = preprocessor.fit_transform(train_clean, y_train_log)
    X_val_proc = preprocessor.transform(val_clean)
    X_test_proc = preprocessor.transform(test_clean)

    # Step G: Export artifacts for downstream training and comparison experiments.
    export_model_artifacts(
        preprocessor=preprocessor,
        X_train_proc=X_train_proc,
        X_val_proc=X_val_proc,
        X_test_proc=X_test_proc,
        y_train=train_clean["ClosePrice"],
        y_val=val_clean["ClosePrice"],
        y_test=test_clean["ClosePrice"],
    )


[Success] Data preprocessing artifacts successfully exported to C:\Users\joeyw\Documents\Fall2026\IDX-Exchange-Data-Science\model_artifacts
